# Gaussian Splatting Colab

Run this notebook with a GPU runtime. Use `WORKFLOW_STEPS` below to run only the stages you need: input preparation from images/video, COLMAP conversion, 3DGS training, rendering/metrics, and optional Drive copy.


In [ ]:
# Runtime and workflow configuration
REPO_URL = "https://github.com/IlhyeonChoo/gaussian-splatting.git"
BRANCH = "main"
WORKDIR = "/content/gaussian-splatting"

SCENE_NAME = "W2_4"  #@param {type:"string"}
MODEL_PATH = ""  #@param {type:"string"}
if not MODEL_PATH:
    MODEL_PATH = f"output/{SCENE_NAME}_colab"

# Choose any comma-separated subset: prepare, colmap, train, render, metrics, copy.
# For a Drive tar.gz that already contains COLMAP results, use prepare,train.
# Aliases such as video, extract_frames, sfm, training, and 3dgs are accepted.
WORKFLOW_STEPS = "prepare,train"  #@param ["prepare", "prepare,train", "prepare,train,copy", "prepare,colmap", "prepare,colmap,train", "colmap", "colmap,train", "train", "render", "metrics", "render,metrics", "prepare,colmap,train,render", "prepare,colmap,train,render,copy"] {allow-input: true}

# Input preparation. INPUT_PATH can be a Drive folder, zip, tar.gz COLMAP scene, video, single image, or uploaded path.
INPUT_PATH = ""  #@param {type:"string"}
if not INPUT_PATH:
    INPUT_PATH = f"/content/drive/MyDrive/3DGS/data/{SCENE_NAME}.tar.gz"
MAX_INPUT_IMAGES = 10000  #@param {type:"integer"}
VIDEO_TARGET_FPS = 2.0  #@param {type:"number"}
VIDEO_SCALE = 1.0  #@param {type:"number"}
VIDEO_WIDTH = 0  #@param {type:"integer"}
VIDEO_HEIGHT = 0  #@param {type:"integer"}
VIDEO_CUSTOM_FORMAT = "jpg"  #@param ["jpg", "png", "webp"]
VIDEO_JPEG_QUALITY = 90  #@param {type:"slider", min:1, max:100, step:1}

# COLMAP conversion.
COLMAP_DEVICE = "auto"  #@param ["auto", "gpu", "cpu"]
COLMAP_PRESET = "video"  #@param ["default", "video", "low-memory", "hard-scene"]
COLMAP_EXTRA_ARGS = ""  #@param {type:"string"}

# Training defaults for Colab. Increase ITERATIONS for final runs.
ITERATIONS = 30000  #@param {type:"integer"}
MAX_TRAIN_CAMERAS = 0  #@param {type:"integer"}
CAMERA_QUALITY_RATIO = 0.7  #@param {type:"number"}
CAMERA_SELECTION_SEED = 42  #@param {type:"integer"}
RESOLUTION = 1  #@param {type:"integer"}
DATA_DEVICE = "cuda"  #@param ["cuda", "cpu"]
EVAL_SPLIT = False  #@param {type:"boolean"}
USE_SPARSE_ADAM = False  #@param {type:"boolean"}
TRAIN_EXTRA_ARGS = ""  #@param {type:"string"}

# Rendering and metrics.
RENDER_EXTRA_ARGS = ""  #@param {type:"string"}
METRICS_EXTRA_ARGS = ""  #@param {type:"string"}

# Persistent storage.
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT_DIR = ""  #@param {type:"string"}
if not DRIVE_PROJECT_DIR:
    DRIVE_PROJECT_DIR = f"/content/drive/MyDrive/3DGS/{SCENE_NAME}"
DRIVE_COPY_OVERWRITE = True  #@param {type:"boolean"}

print(f"Configured scene: {SCENE_NAME}")
print(f"Input path: {INPUT_PATH}")
print(f"Model path: {MODEL_PATH}")
print(f"Drive output path: {DRIVE_PROJECT_DIR}")


In [ ]:
# Check GPU runtime.
import subprocess

subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# Clone/update the repository and install Colab dependencies.
import os
import subprocess
from pathlib import Path

repo_path = Path(WORKDIR)
if repo_path.exists() and (repo_path / ".git").exists():
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "pull", "--ff-only", "origin", BRANCH], check=True)
elif repo_path.exists():
    raise RuntimeError(f"WORKDIR exists but is not a git repository: {repo_path}")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)], check=True)

os.chdir(repo_path)
try:
    subprocess.run(["bash", "colab/setup_colab.sh"], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print(f"Error running setup script: {e}")
    print(f"Stdout:\\n{e.stdout}")
    print(f"Stderr:\\n{e.stderr}")
    raise


In [ ]:
# Mount Google Drive for input/output persistence.
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"Drive project directory: {DRIVE_PROJECT_DIR}")
else:
    print("Drive mount skipped.")


## Optional Upload

Run this cell only when you want to upload images, a zip, a tar.gz COLMAP scene, or a video from your computer. It sets `INPUT_PATH`; the selected workflow cell still controls whether the prepare step runs.


In [ ]:
# Optional: upload images, a zip, a tar.gz COLMAP scene, or a video from your computer.
UPLOAD_INPUT = False  #@param {type:"boolean"}

if UPLOAD_INPUT:
    import os
    import shutil
    import sys
    from pathlib import Path

    from google.colab import files

    os.chdir(WORKDIR)
    if WORKDIR not in sys.path:
        sys.path.insert(0, WORKDIR)

    from colab.colab_utils import IMAGE_EXTS

    uploaded = files.upload()
    upload_dir = Path("/content/gaussian_splatting_uploads")
    if upload_dir.exists():
        shutil.rmtree(upload_dir)
    upload_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []
    for filename, content in uploaded.items():
        destination = upload_dir / filename
        destination.write_bytes(content)
        saved_paths.append(destination)

    if saved_paths:
        all_images = all(path.suffix.lower() in IMAGE_EXTS for path in saved_paths)
        source = upload_dir if all_images else saved_paths[0]
        INPUT_PATH = str(source)
        print(f"INPUT_PATH set to: {INPUT_PATH}")
    else:
        print("No files uploaded.")
else:
    print(f"Upload skipped. Current INPUT_PATH: {INPUT_PATH}")


## Run Selected Workflow

Run this cell after setup. It executes the stages listed in `WORKFLOW_STEPS` in order and skips the rest.


In [ ]:
import os
import shlex
import sys
from pathlib import Path

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

from colab.colab_utils import (
    copy_model_to_drive,
    normalize_workflow_steps,
    prepare_uploaded_path,
    run_colmap_conversion,
    run_metrics,
    run_rendering,
    run_training,
    verify_colmap_scene,
    verify_scene_input,
)


def optional_positive_int(value):
    value = int(value)
    return None if value <= 0 else value


steps = normalize_workflow_steps(WORKFLOW_STEPS)
print(f"Workflow steps: {steps}")

if "prepare" in steps:
    if not INPUT_PATH:
        raise RuntimeError("Set INPUT_PATH or run the upload cell before the prepare step.")
    scene_path = prepare_uploaded_path(
        INPUT_PATH,
        SCENE_NAME,
        max_images=MAX_INPUT_IMAGES,
        root=WORKDIR,
        target_fps=VIDEO_TARGET_FPS,
        scale=VIDEO_SCALE,
        width=optional_positive_int(VIDEO_WIDTH),
        height=optional_positive_int(VIDEO_HEIGHT),
        custom_format=VIDEO_CUSTOM_FORMAT,
        jpeg_quality=VIDEO_JPEG_QUALITY,
    )
    print(f"[DONE] Prepared scene: {scene_path}")
else:
    print("[SKIP] prepare")

if "colmap" in steps:
    verify_scene_input(SCENE_NAME, WORKDIR)
if "train" in steps:
    verify_colmap_scene(SCENE_NAME, WORKDIR)

if "colmap" in steps:
    run_colmap_conversion(
        SCENE_NAME,
        root=WORKDIR,
        colmap_device=COLMAP_DEVICE,
        colmap_preset=COLMAP_PRESET,
        extra_args=shlex.split(COLMAP_EXTRA_ARGS),
    )
else:
    print("[SKIP] colmap")

if "train" in steps:
    run_training(
        SCENE_NAME,
        MODEL_PATH,
        root=WORKDIR,
        iterations=ITERATIONS,
        max_train_cameras=MAX_TRAIN_CAMERAS,
        camera_quality_ratio=CAMERA_QUALITY_RATIO,
        camera_selection_seed=CAMERA_SELECTION_SEED,
        resolution=RESOLUTION,
        data_device=DATA_DEVICE,
        eval_split=EVAL_SPLIT,
        use_sparse_adam=USE_SPARSE_ADAM,
        extra_args=shlex.split(TRAIN_EXTRA_ARGS),
    )
else:
    print("[SKIP] train")

if "render" in steps:
    run_rendering(MODEL_PATH, root=WORKDIR, extra_args=shlex.split(RENDER_EXTRA_ARGS))
else:
    print("[SKIP] render")

if "metrics" in steps:
    run_metrics(MODEL_PATH, root=WORKDIR, extra_args=shlex.split(METRICS_EXTRA_ARGS))
else:
    print("[SKIP] metrics")

if "copy" in steps:
    copy_model_to_drive(MODEL_PATH, DRIVE_PROJECT_DIR, overwrite=DRIVE_COPY_OVERWRITE)
else:
    print("[SKIP] copy")
